# Silver Layer — Orders Transformation

This notebook transforms data from the Bronze orders table and writes the clean output to the Silver layer.

## What This Notebook Does

* Reads from `bronze.orders` and applies cleaning transformations (null filter, deduplication, status standardization, audit timestamp).
* Adds per-pair boolean columns to mark rows where the order event sequence is violated (e.g., `order_approved_at > order_delivered_carrier_date`).
* Splits the data into two paths: rows with violations are written to `quarantine.orders` with a `violation_reasons` column; clean rows are written to `silver.orders`.
* Runs all four DQ checks on the clean DataFrame as a final sanity step before writing to Silver.

## Input

`retail_lakehouse.bronze.orders`

## Output

* Clean rows → `retail_lakehouse.silver.orders`
* Violating rows → `retail_lakehouse.quarantine.orders` (with violation reasons)

## Design Decisions

* **Quarantine bad rows instead of dropping or stopping the pipeline.** When a row violates the event sequence, it goes to a separate `quarantine.orders` table along with a column saying which rule it failed. The clean rows go to Silver. This way Silver stays clean, but we still have a record of which rows had problems and why.

* **Labeller for the split, gate as a final check.** `identify_event_sequence_violations` only marks rows — it doesn't stop the pipeline. The notebook uses it to split clean from violating rows. After the split, `check_event_sequence` is called on the clean DataFrame to confirm no violations slipped through. The split function does the routing; the check function makes sure the routing worked.

* **Row count tolerance set wide on purpose.** Some rows will go to quarantine, so Silver will always have slightly fewer rows than Bronze. The 70–100% range allows for that without false alarms.

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
%run ../Utilities/silver_dq_checks


# Silver Layer — Data Quality Checks

This notebook defines reusable data quality check functions used across the Silver layer transformation notebooks.

Each function validates a specific property of a transformed DataFrame before it is written to a Silver Delta table. On success the function prints a PASSED message; on failure it raises a `ValueError` with details, stopping the pipeline.

## Checks Defined Here

* `check_not_null` — verifies that specified columns have no null or blank values
* `check_unique` — verifies that the given key (single or composite) has no duplicates
* `check_row_count` — verifies that the Silver row count falls within an expected percentage range of the Bronze source

## How To Use

Import these functions into a Silver transformation notebook by running:
​```
%run ../Utilities/silver_dq_checks
​```
Then call the functions on the transformed DataFrame before writing to Silver.

### check_not_null
Validates that the specified columns contain no null or blank/whitespace values.

In [0]:
catalog_name = 'retail_lakehouse'
bronze_schema = 'bronze'
silver_schema = 'silver'
quarantine_schema = 'quarantine'

In [0]:
%sql
DROP TABLE IF EXISTS retail_lakehouse.silver.orders

### check_unique
Validates that the specified key columns produce unique rows. Supports single-column or composite keys.

## Transformations
Filtering out nulls, dropping duplicates, trimming and lowercasing.

In [0]:
df_orders = spark.read.table(f'{catalog_name}.{bronze_schema}.orders')
transformed_orders_df =df_orders.filter(col('order_id').isNotNull() & col('customer_id').isNotNull() 
                                 & col('order_status').isNotNull() 
                                 & col('order_purchase_timestamp').isNotNull())\
                                     .dropDuplicates(['order_id'])\
                                         .withColumn('order_status', lower(trim(col('order_status'))))\
                                             .withColumn('silver_processed_at', current_timestamp())


### check_row_count
Validates that the Silver row count is within an acceptable percentage range of the Bronze source row count.


###check_event_sequence

Validates that the timestamps for orders is in correct sequence.

###identify_event_sequence_violations

Add audit columns and flag the records with the boolean which violates the rule.


## Identify event sequence violations 
- Executes identify_event_sequence_violations function which add the audit columns for violating the rule and also add violation reason column. 
- Lastly, split the dataframe into quarantine dataframe and clean dataframe

In [0]:
ordered_timestamps = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date'
]

violation_columns = [
    '_violates_order_purchase_timestamp_order_approved_at',
    '_violates_order_approved_at_order_delivered_carrier_date',
    '_violates_order_delivered_carrier_date_order_delivered_customer_date'
]

df_violations = identify_event_sequence_violations(transformed_orders_df,ordered_timestamps)

transformed_final = df_violations.withColumn(
    'violation_reasons',
    concat_ws(
        '; ',
        when(col('_violates_order_purchase_timestamp_order_approved_at'),
             lit('order_purchase_timestamp > order_approved_at')),
        when(col('_violates_order_approved_at_order_delivered_carrier_date'),
             lit('order_approved_at > order_delivered_carrier_date')),
        when(col('_violates_order_delivered_carrier_date_order_delivered_customer_date'),
             lit('order_delivered_carrier_date > order_delivered_customer_date'))
    )
)

filter_conditions = (col(violation_columns[0])) | (col(violation_columns[1]))| (col(violation_columns[2]))

df_quarantine = transformed_final.filter(filter_conditions)

df_clean = transformed_final.filter(~filter_conditions).drop(*violation_columns,'violation_reasons')

In [0]:
df_clean.printSchema()


root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)
 |-- ingestion_time: timestamp (nullable = true)
 |-- source_file: string (nullable = true)
 |-- silver_processed_at: timestamp (nullable = false)



## Writing quarantine dataframe to quarantine schema


In [0]:
df_quarantine.write.format('delta').mode('overwrite').saveAsTable(f'{catalog_name}.{quarantine_schema}.orders')

%md
## Data Quality Checks on Clean DataFrame
Running all four DQ checks on the clean DataFrame before writing to Silver. The event sequence check acts as a sanity verification — it should pass because violating rows have already been moved to quarantine.

In [0]:
check_not_null(df_clean,['order_id','customer_id','order_status','order_purchase_timestamp'])
check_unique(df_clean,['order_id'])
check_row_count(df_clean,f'{catalog_name}.{bronze_schema}.orders',85,100)
check_event_sequence(df_clean,ordered_timestamps)

check_not_null: PASSED
check_unique: PASSED
check_row_count: PASSED
check_event_sequence: PASSED


## Writing clean dataframe to silver schema

In [0]:
df_clean.write.format('delta').mode('overwrite').saveAsTable(f'{catalog_name}.{silver_schema}.orders')


In [0]:
print(f"Bronze orders:{spark.read.table(f'{catalog_name}.{bronze_schema}.orders').count()}")
print(f"Silver orders:{spark.read.table(f'{catalog_name}.{silver_schema}.orders').count()}")
print(f"Quarantine orders:{spark.read.table(f'{catalog_name}.{quarantine_schema}.orders').count()}")

Bronze orders:99441
Silver orders:98059
Quarantine orders:1382
